# S&P 500 Data Preparation & Exploratory Data Analysis (EDA)

In [1]:
# Import the needed libraries
import numpy as np 
import pandas as pd 
import yfinance as yf
# Setting Plotly backend plotting for pandas 
pd.options.plotting.backend = 'plotly'
import plotly.io as pio
pio.templates.default = 'plotly_dark'
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import holidays 
from statsmodels.tsa.stattools import adfuller
import os
from IPython.display import Markdown, display
from scipy import stats
import pandas_market_calendars as mcal



## Load Data 

In [2]:
# import raw data from yfinance
# === DATA DOWNLOAD ===

# End date set dynamically to today's date.
# Ensures the pipeline always pulls the most recent available data.
# The download end date is logged alongside the dataset for audit purposes —
# the decision log references this timestamp to confirm which session
# informed the weekly risk signal.

end_date = pd.Timestamp('today').normalize()

raw = yf.download('^GSPC', start='2000-01-01', end=end_date)

print(f"Dataset downloaded: 2000-01-01 to {end_date.date()}")
print(f"Total sessions: {len(raw)}")

[*********************100%***********************]  1 of 1 completed

Dataset downloaded: 2000-01-01 to 2026-06-29
Total sessions: 6660


In [3]:
# check our loaded data 
raw

Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2000-01-03,1455.219971,1478.000000,1438.359985,1469.250000,931800000
2000-01-04,1399.420044,1455.219971,1397.430054,1455.219971,1009000000
2000-01-05,1402.109985,1413.270020,1377.680054,1399.420044,1085500000
2000-01-06,1403.449951,1411.900024,1392.099976,1402.109985,1092300000
2000-01-07,1441.469971,1441.469971,1400.729980,1403.449951,1225200000
...,...,...,...,...,...
2026-06-22,7472.790039,7530.009766,7460.009766,7500.439941,5976740000
2026-06-23,7365.459961,7424.169922,7347.600098,7366.509766,5640530000


## No duplicate index check


In [4]:
# yfinance occasionally returns duplicate dates
assert raw.index.duplicated().sum() == 0, "Duplicate dates found in index"

### 📊 Understanding OHLCV Market Data

Each row in the dataset represents a **single trading day** in the market.


**Open**  
The first recorded price at the start of the trading session (**9:30 AM EST**).  
It often reflects **overnight news, earnings, macroeconomic developments, and shifts in investor sentiment** while markets were closed.


**High & Low**  
The **maximum (High)** and **minimum (Low)** prices reached during the trading session.

- Together, they define the **intraday range**
- This provides insight into **market volatility and uncertainty**
- Larger ranges typically indicate **strong reactions to new information or increased trading activity**


**Close** ⭐ *(Most Important)*  
The final price recorded at the end of the trading session (**4:00 PM EST**).

- Represents the market’s **final consensus on valuation**
- The **primary variable used in financial analysis and modeling**
- Most time series models are built using:
  - Closing prices, or
  - Returns derived from closing prices


**Volume**  
Represents the level of trading activity during the day.

- For indices like the **S&P 500**, volume is **not directly observed**
- It is **derived or provided by the data source**
- As a result:
  - It may be **less reliable**
  - It is **not typically used as a primary feature in modeling**

## Data Inspection 

In [5]:
# Check the DataSet shape
raw.shape

(6660, 5)

In [6]:
# Check Datatime index
type(raw.index)

pandas.DatetimeIndex

In [7]:
# Check for missing values 
raw.isnull().sum()

Price   Ticker
Close   ^GSPC     0
High    ^GSPC     0
Low     ^GSPC     0
Open    ^GSPC     0
Volume  ^GSPC     0
dtype: int64

In [8]:
# Check Data description 
summary = raw.describe()
summary

Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
count,6660.000000,6660.000000,6660.000000,6660.000000,6.660000e+03
mean,2355.629839,2368.618677,2340.889964,2355.259558,3.458858e+09
std,1576.063618,1583.023288,1567.891554,1575.712233,1.533392e+09
min,676.530029,695.270020,666.789978,679.280029,0.000000e+00
25%,1214.462463,1221.437500,1206.982544,1214.419952,2.356682e+09
50%,1561.985046,1565.085022,1551.859985,1561.434998,3.551930e+09
75%,2971.800110,2984.145081,2954.202393,2968.750061,4.309428e+09
max,7609.779785,7620.899902,7582.990234,7605.310059,1.145623e+10


In [9]:
len(raw)

6660

In [10]:

display(Markdown(
    f"The dataset contains **~{len(raw):,} daily observations**, providing a sufficiently large sample for statistical analysis and modeling."
))


The dataset contains **~6,660 daily observations**, providing a sufficiently large sample for statistical analysis and modeling.

In [11]:
# Inspect your current column structure
raw.columns

MultiIndex([( 'Close', '^GSPC'),
            (  'High', '^GSPC'),
            (   'Low', '^GSPC'),
            (  'Open', '^GSPC'),
            ('Volume', '^GSPC')],
           names=['Price', 'Ticker'])

In [12]:
# Flatten multi-level columns

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# Validate the dataset flatten
print("Columns after flattening:", raw.columns.tolist())


Columns after flattening: ['Open', 'High', 'Low', 'Close', 'Volume']


In [13]:
raw.head()

Price,Open,High,Low,Close,Volume
Date,,,,,
2000-01-03,1469.250000,1478.000000,1438.359985,1455.219971,931800000
2000-01-04,1455.219971,1455.219971,1397.430054,1399.420044,1009000000
2000-01-05,1399.420044,1413.270020,1377.680054,1402.109985,1085500000
2000-01-06,1402.109985,1411.900024,1392.099976,1403.449951,1092300000
2000-01-07,1403.449951,1441.469971,1400.729980,1441.469971,1225200000


### 🧩 Column Structure Adjustment: Flattening Multi-Level Headers

The dataset initially contains **multi-level column headers** due to the structure returned by the data source (*yfinance*), where each column is paired with the ticker symbol.

While this format is useful when working with **multiple assets**, it introduces unnecessary complexity in a **single-index context**.


**Why this matters**

- Complicates column selection and referencing  
- Increases the risk of errors in:
  - Feature engineering  
  - Data transformations  
  - Model input preparation  


**Action Taken**

- The multi-level columns were **flattened to a single level**
- This ensures:
  - Simpler column access  
  - Cleaner data manipulation  
  - Greater consistency across the pipeline  


**Key Insight**

> Standardizing the dataset structure early improves **robustness, readability, and maintainability** of the data pipeline.

In [14]:
display(Markdown(
    f"The minimum value of the **Volume** column is **{raw['Volume'].min():,.0f}**, which is unexpected for a confirmed trading day.\n\n"
    f"This warrants further investigation to assess whether it represents a data integrity issue or a legitimate market condition."
))

The minimum value of the **Volume** column is **0**, which is unexpected for a confirmed trading day.

This warrants further investigation to assess whether it represents a data integrity issue or a legitimate market condition.

# Zero Volume Observation

In [15]:
zero_vol = raw[raw['Volume'] == 0].copy()
sessions_before = len(raw)
 
print(f"Zero-volume observations detected: {len(zero_vol)}")
if len(zero_vol) > 0:
    print(f"Affected date(s): {[str(d.date()) for d in zero_vol.index]}")

Zero-volume observations detected: 1
Affected date(s): ['2023-05-24']


In [16]:

# This pipeline is calibrated for a single known anomaly.
# If yfinance revises historical data or a new anomaly appears,
# this assertion will fail immediately rather than silently
# removing multiple observations.
 
if len(zero_vol) > 1:
    raise ValueError(
        f"Expected at most one zero-volume observation; found {len(zero_vol)}. "
        "Inspect zero_vol before proceeding."
    )

In [17]:
# Timedelta does not guarantee 5 trading sessions — weekends
# can reduce the window to 4 observations. iloc gives exactly
# 5 observations on each side.
 
if len(zero_vol) > 0:
    idx = zero_vol.index[0]
    loc = raw.index.get_loc(idx)
 
    # Clamp to valid index bounds
    window = raw.iloc[max(0, loc - 5) : min(len(raw), loc + 6)]
 
    print("5 trading sessions before and after zero-volume observation")
    display(window[['Close', 'Volume']])
 

5 trading sessions before and after zero-volume observation


Price,Close,Volume
Date,,
2023-05-17,4158.770020,4039080000
2023-05-18,4198.049805,3980500000
2023-05-19,4191.979980,4041900000
2023-05-22,4192.629883,3728520000
2023-05-23,4145.580078,4155320000
2023-05-24,4115.240234,0
2023-05-25,4151.279785,4147760000
2023-05-26,4205.450195,3715460000
2023-05-30,4205.520020,4228510000


In [18]:
# holidays.US() tests against the federal holiday calendar.
# The NYSE does not follow the federal calendar exactly —
# Good Friday, Juneteenth (since 2022), and national days of
# mourning are observed by the exchange but not federal statute.
# pandas_market_calendars uses the NYSE schedule directly and
# is more defensible for a market data pipeline.

if len(zero_vol) > 0:
    zero_date = zero_vol.index[0]
    nyse = mcal.get_calendar('NYSE')
 
    schedule = nyse.schedule(
        start_date=zero_date.strftime('%Y-%m-%d'),
        end_date=zero_date.strftime('%Y-%m-%d')
    )
 
    if schedule.empty:
        market_status = "NYSE was closed on this date."
        holiday_note = "falls on a NYSE non-trading day"
    else:
        market_status = "NYSE was open on this date — expected trading session."
        holiday_note = "does not fall on a NYSE holiday — treated as a data integrity error"
 
    print(f"{zero_date.date()} | {market_status}")

2023-05-24 | NYSE was open on this date — expected trading session.


In [19]:

# The chart is built from raw — the full dataset before the
# anomaly is removed. This ensures the zero-volume observation
# is visible on the series. After raw is filtered in Cell 6,
# the point no longer exists in the index.
 
fig = go.Figure()
 
fig.add_trace(
    go.Scatter(
        x=raw.index,
        y=raw['Volume'],
        mode='lines',
        name='Volume',
        line=dict(width=1, color='#5DCAA5')
    )
)
 
if len(zero_vol) > 0:
    anomaly_date = zero_vol.index[0]
    anomaly_date_str = str(anomaly_date.date())  # Plotly cannot serialise pd.Timestamp directly
    anomaly_volume = zero_vol.loc[anomaly_date, 'Volume']  # 0 — from zero_vol, not raw
 
    fig.add_trace(
        go.Scatter(
            x=[anomaly_date_str],
            y=[anomaly_volume],
            mode='markers',
            name='Zero-volume observation',
            marker=dict(size=12, color='#E24B4A', symbol='x')
        )
    )
 
    fig.add_annotation(
        x=anomaly_date_str,
        y=anomaly_volume,
        text=(
            f"Zero volume: {anomaly_date.date()}"
            "<br>Expected NYSE session"
            "<br>Removed before modelling"
        ),
        showarrow=True,
        arrowhead=2,
        ax=130,
        ay=-80,
        font=dict(size=11),
        bgcolor='rgba(30,30,30,0.75)',
        bordercolor='#E24B4A',
        borderwidth=1
    )
 
fig.update_layout(
    title='S&P 500 volume — anomaly detection (pre-removal)',
    xaxis_title='Date',
    yaxis_title='Volume',
    template='plotly_dark',
    height=500,
    width=1400,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
 
fig.show()
 
fig.write_image(
    '../reports/figures/sp500_volume_anomaly.png',
    width=1200,
    height=627,
    scale=2
)

## Zero Volume Investigation

In [20]:

raw = raw[raw['Volume'] > 0]
sessions_after = len(raw)
 
display(Markdown(f"""
## Data integrity: zero-volume observation
 
During data validation, {len(zero_vol)} zero-volume observation(s) were
identified in the downloaded dataset.
 
The S&P 500 index is published for every regular NYSE trading session. A
zero value in the downloaded Volume field is inconsistent with normal market
activity and is treated as a data integrity issue, not a valid market
observation.
 
The affected observation occurred on **{zero_date.date()}**. Calendar
verification against the NYSE schedule confirmed that {holiday_note}.
 
The observation was removed before calculating returns, rolling statistics,
or fitting any forecasting models. A single erroneous record propagating
through feature engineering and model estimation could bias volatility
estimates and invalidate downstream diagnostics.
 
---
 
Sessions before cleaning: **{sessions_before:,}**  
Observations removed: **{len(zero_vol)}**  
Sessions after cleaning: **{sessions_after:,}**  
Final date range: **{raw.index[0].date()} to {raw.index[-1].date()}**
"""))


## Data integrity: zero-volume observation

During data validation, 1 zero-volume observation(s) were
identified in the downloaded dataset.

The S&P 500 index is published for every regular NYSE trading session. A
zero value in the downloaded Volume field is inconsistent with normal market
activity and is treated as a data integrity issue, not a valid market
observation.

The affected observation occurred on **2023-05-24**. Calendar
verification against the NYSE schedule confirmed that does not fall on a NYSE holiday — treated as a data integrity error.

The observation was removed before calculating returns, rolling statistics,
or fitting any forecasting models. A single erroneous record propagating
through feature engineering and model estimation could bias volatility
estimates and invalidate downstream diagnostics.

---

Sessions before cleaning: **6,660**  
Observations removed: **1**  
Sessions after cleaning: **6,659**  
Final date range: **2000-01-03 to 2026-06-26**


In [21]:
# Confirm no zero-volume rows remain
raw[raw['Volume'] == 0]

Price,Open,High,Low,Close,Volume
Date,,,,,


In [22]:
# === DATA INTEGRITY: INCOMPLETE SESSION HANDLING ===

# Original approach — unconditional drop of the last row
# Rationale: the latest observation may represent partial or delayed data
# depending on when the API request is made. Removing it prevents incomplete
# or partially recorded observations from entering the pipeline and avoids
# bias in return calculations and volatility estimates.
#
# raw = raw.iloc[:-1]  # Enforce use of fully confirmed data only (T-1)

# Updated approach — conditional drop
# The live signal generation requirement changes the constraint. The system
# produces a weekly risk signal every Monday to inform that day's DCA decision.
# That signal must reflect Friday's complete close — the most recent finalised
# session. An unconditional drop would silently remove Friday's data when the
# pipeline runs over the weekend or on Monday morning, generating a signal from
# Thursday's figures without any indication something was wrong.
#
# The pipeline now drops the last row only when it represents today's
# incomplete session, preserving Friday's complete close for Monday
# signal generation.

today = pd.Timestamp('today').normalize()

if raw.index[-1] >= today:
    raw = raw.iloc[:-1]
    print(f"Incomplete session detected and removed: {today.date()}")
else:
    print(f"Last confirmed session retained: {raw.index[-1].date()}")

Last confirmed session retained: 2026-06-26


In [23]:
# Confirm the last row
raw.tail()

Price,Open,High,Low,Close,Volume
Date,,,,,
2026-06-22,7500.439941,7530.009766,7460.009766,7472.790039,5976740000
2026-06-23,7366.509766,7424.169922,7347.600098,7365.459961,5640530000
2026-06-24,7370.879883,7428.060059,7336.819824,7358.220215,5994860000
2026-06-25,7404.910156,7419.080078,7323.500000,7357.490234,5565760000
2026-06-26,7312.740234,7392.950195,7294.180176,7354.020020,9105600000


In [24]:
# Visualize Close Price over time
fig = raw['Close'].plot(title='S&P 500 Close Price (2000–Present)')
fig.update_layout(
    showlegend=False,
    yaxis_title='S&P 500',
    xaxis_title='Date'
)
fig.show()
fig.write_image(
    "../reports/figures/sp500_close_price.png",
    width=1400,
    height=700,
    scale=2
)

# Data stationarity assumptions

To assess suitability for **financial time series modeling**, raw closing prices are transformed into **log returns**.

Raw prices are typically **non-stationary**, exhibiting trends and compounding effects that violate the assumptions of many statistical models. Transforming prices into log returns removes these effects and results in a series with more stable statistical properties.

Log returns are widely used in quantitative finance due to their **additivity over time** and suitability for modeling relative price changes.

The resulting series is then analyzed to evaluate its statistical behavior prior to model selection.

## Achieve stationarity
| **Property**             | **Issue**           | **Fix**               | **Test** |
| ------------------------ | ------------------- | --------------------- | -------- |
| Stationarity in variance | Changing volatility | Log transform / GARCH | ARCH     |
| Stationarity in mean     | Trend / Drift       | Differencing          | ADF      |


In [25]:
# We create a new column and calculate the daily log Returns

raw['log_returns'] = np.log(raw['Close']/ raw['Close'].shift(1))



In [26]:
# Double check our results

raw[['Close', 'log_returns']].head(10)

Price,Close,log_returns
Date,,
2000-01-03,1455.219971,NaN
2000-01-04,1399.420044,-0.039099
2000-01-05,1402.109985,0.001920
2000-01-06,1403.449951,0.000955
2000-01-07,1441.469971,0.026730
2000-01-10,1457.599976,0.011128
2000-01-11,1438.560059,-0.013149
2000-01-12,1432.250000,-0.004396
2000-01-13,1449.680054,0.012096


In [27]:
# I drop the first row as we can't compute daily returns for the first day of our data 
raw = raw.dropna(subset=['log_returns'])

# To confirm that all NaN values dropped:
raw['log_returns'].isna().sum()

np.int64(0)

In [28]:
raw['log_returns'].describe()

count    6658.000000
mean        0.000243
std         0.012163
min        -0.127652
25%        -0.004730
50%         0.000642
75%         0.005878
max         0.109572
Name: log_returns, dtype: float64

In [29]:

daily_vol = raw['log_returns'].std() * 100
display(Markdown(
    f"The mean of log returns is close to zero, which is consistent with empirical findings in financial markets. However, this alone does not imply predictability or stationarity.\n\n"
    f"However, this alone does **not confirm stationarity**. Formal statistical tests (e.g., Augmented Dickey-Fuller) are required to assess whether the series satisfies stationarity assumptions.\n\n"
    f"The standard deviation of log returns represents **daily volatility**, indicating the typical magnitude of price fluctuations. In this dataset, it is approximately **{daily_vol:.2f}% per day**, which is consistent with historical equity market behavior."
))


The mean of log returns is close to zero, which is consistent with empirical findings in financial markets. However, this alone does not imply predictability or stationarity.

However, this alone does **not confirm stationarity**. Formal statistical tests (e.g., Augmented Dickey-Fuller) are required to assess whether the series satisfies stationarity assumptions.

The standard deviation of log returns represents **daily volatility**, indicating the typical magnitude of price fluctuations. In this dataset, it is approximately **1.22% per day**, which is consistent with historical equity market behavior.

In [30]:
# We check the min and max dates for the daily returns :
# Find the actual dates of extreme moves
raw['log_returns'].nsmallest(5)

Date
2020-03-16   -0.127652
2020-03-12   -0.099945
2008-10-15   -0.094695
2008-12-01   -0.093537
2008-09-29   -0.092190
Name: log_returns, dtype: float64

In [31]:
raw['log_returns'].nlargest(5)

Date
2008-10-13    0.109572
2008-10-28    0.102457
2025-04-09    0.090895
2020-03-24    0.089683
2020-03-13    0.088808
Name: log_returns, dtype: float64

### 📉 Extreme Market Movements

To better understand tail behavior, the largest positive and negative daily returns were examined.

These extreme observations **exceed typical daily movements by multiple standard deviations**, indicating periods of significant market stress.


**Largest Gains**

- **2008 Financial Crisis** — Multiple large upward moves driven by bailout announcements and relief rallies  
- **March 2020 (COVID-19)** — Sharp rebounds during periods of extreme volatility  
- **April 2025** — Elevated volatility linked to major macroeconomic policy announcements  


**Largest Losses**

- **March 16, 2020** — Largest single-day drop (~-12.7%), following global COVID-19 lockdown escalation  
- **March 12, 2020** — WHO pandemic declaration triggered sharp market decline  
- **2008 Financial Crisis** — Sustained drawdowns during systemic financial instability  


These observations highlight the presence of **fat tails** and **extreme risk events**, which are key characteristics of financial return series.

In [32]:
# Create figure with secondary y-axis for rolling volatility
fig = make_subplots(specs=[[{"secondary_y": True}]])

# 1. Daily log returns (main trace)
fig.add_trace(
    go.Scatter(
        x=raw.index,
        y=raw['log_returns'],
        name='Daily Log Returns',
        line=dict(color='#1E88E5', width=1.2),
        opacity=0.85
    ),
    secondary_y=False
)

# 2. 21-day rolling volatility (highlighted)
rolling_vol = raw['log_returns'].rolling(window=21).std() * 100  # in percentage

fig.add_trace(
    go.Scatter(
        x=raw.index,
        y=rolling_vol,
        name='21-day Rolling Volatility (%)',
        line=dict(color='#D32F2F', width=2.5),
        opacity=0.9
    ),
    secondary_y=True
)

# Layout improvements
fig.update_layout(
    title='S&P 500 Daily Log Returns vs 21-day Rolling Volatility (2000–Present)',
    template='plotly_dark',
    height=720,
    width=1450,
    hovermode='x unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5
    ),
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.update_yaxes(title_text='Daily Log Return', secondary_y=False, gridcolor='rgba(255,255,255,0.08)')
fig.update_yaxes(title_text='Rolling Volatility (%)', secondary_y=True, gridcolor='rgba(255,255,255,0.08)')

# Optional: Add vertical spans for major events (highly recommended)
fig.add_vrect(x0='2008-09-01', x1='2009-06-01', fillcolor='red', opacity=0.1, layer='below', line_width=0)
fig.add_vrect(x0='2020-02-01', x1='2020-06-01', fillcolor='red', opacity=0.1, layer='below', line_width=0)

fig.show()

# Export high-quality image
fig.write_image(
    "../reports/figures/sp500_log_returns_with_vol.png",
    width=1450,
    height=720,
    scale=2.5   
)

### 📈 Volatility Clustering

The log return time series exhibits **volatility clustering**, where periods of high volatility are followed by further high volatility, and periods of relative calm tend to persist.


**Notable High-Volatility Periods**

- **2008 Financial Crisis**  
- **2020 COVID-19 market shock**  
- Elevated volatility observed in **2025–2026**


**Low-Volatility Regime**

- The period between **2013–2019** shows relatively stable and lower volatility  


**Interpretation**

Periods of elevated volatility are characterized by larger absolute returns, indicating increased market uncertainty and risk.

This behavior reflects a key stylized fact of financial markets and indicates that volatility is **time-dependent rather than constant**.

As a result, the assumption of constant variance is violated, which has important implications for modeling. It motivates the use of models that can account for time-varying volatility, such as rolling statistics, GARCH-type models, or more flexible machine learning approaches.

To further quantify this behavior, rolling measures of volatility are computed in the next step.

In [33]:
extremes = raw['log_returns'].nsmallest(1)
extreme_loss = extremes.iloc[0]

extremes_high = raw['log_returns'].nlargest(1)
extreme_gain = extremes_high.iloc[0]


In [34]:
fig = raw['log_returns'].plot(title='Distribution of S&P 500 Daily Log Returns', kind='hist')
fig.update_layout(
    showlegend=False,
    yaxis_title='Observations',
    xaxis_title='Log Return'
)
# Add annotation for the March 2020 COVID Crash
fig.add_annotation(
    x=extreme_loss,             # The ~-12.7% log return identified in your analysis
    y=raw['log_returns'].value_counts().max() * 0.01,                  
    text="March 2020 (COVID-19 Shock)",
    showarrow=True,
    arrowhead=1,
    ax=-40,               # Horizontal offset for the text box
    ay=-40                # Vertical offset for the text box
)
# Add annotation for the 2008 Financial Crisis Relief Rally
fig.add_annotation(
    x=extreme_gain,              # Use the exact value from your nlargest(5) results
    y=raw['log_returns'].value_counts().max() * 0.01,                  # Keep y low on the Frequency axis for the tail
    text="2008 Financial Crisis (Extreme Positive Return)",
    showarrow=True,
    arrowhead=1,
    ax=40,                # Positive offset to place text to the right of the arrow
    ay=-40
)
# Add mean / volatility annotation
fig.add_annotation(
    x=0.06,                     # Position toward right tail
    y=raw['log_returns'].value_counts().max() * 0.20,
    text=(
        "Mean ≈ 0.0003"
        "<br>Std Dev ≈ 1.2%"
    ),
    showarrow=False,
    bgcolor="rgba(0,0,0,0.6)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=12)
)
fig.show()
fig.write_image(
    "../reports/figures/sp500_return_distribution.png",
    width=1400,
    height=700,
    scale=2
)

### 📊 Distribution of Log Returns

The histogram of log returns provides insight into the **distribution of daily market movements**.


**Key Observations**

- The distribution is centered around **zero**, consistent with financial return series  
- The presence of **extreme values** on both sides indicates **fat tails**  
- These tail events correspond to periods of significant market stress  


**Interpretation**

Unlike a normal distribution, financial returns exhibit **higher probability of extreme events**, which is a well-documented characteristic in quantitative finance.

This has important implications for risk modeling, as standard Gaussian assumptions may underestimate tail risk.

The modeling approach focuses on capturing **patterns in return magnitude and volatility**, rather than explaining the underlying economic causes of these movements.

In practice, financial models are typically built on **returns rather than raw prices**, as returns exhibit more stable statistical properties and are more suitable for time series analysis.

# Stationarity Test
##  Augmented Dickey-Fuller (ADF) test

In [35]:
def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    
    print(f'ADF Test for {name}')
    print('-' * 30)
    print(f'ADF Statistic : {result[0]:.4f}')
    print(f'P-Value       : {result[1]:.6f}')
    if result[1] < 0.05:
        print("✅ Reject null hypothesis — series is stationary")
    else:
        print('Fail to Reject null hypothesis - series is Not Stationary')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'   {key}: {value:.4f}')




In [36]:
adf_test(raw['log_returns'], 'Log Returns')

ADF Test for Log Returns
------------------------------
ADF Statistic : -19.6247
P-Value       : 0.000000
✅ Reject null hypothesis — series is stationary
Critical Values:
   1%: -3.4313
   5%: -2.8620
   10%: -2.5670


In [37]:

adf_result = adfuller(raw['log_returns'].dropna())
display(Markdown(f"""
### 📊 Stationarity Test: Augmented Dickey-Fuller (ADF)

**Test Results**

- **ADF Statistic:** {adf_result[0]:.2f}
- **P-Value:** {adf_result[1]:.6f}
- **Critical Values:**
  - 1%: {adf_result[4]['1%']:.2f}
  - 5%: {adf_result[4]['5%']:.2f}
  - 10%: {adf_result[4]['10%']:.2f}
"""))



### 📊 Stationarity Test: Augmented Dickey-Fuller (ADF)

**Test Results**

- **ADF Statistic:** -19.62
- **P-Value:** 0.000000
- **Critical Values:**
  - 1%: -3.43
  - 5%: -2.86
  - 10%: -2.57


In [38]:

# Stationarity Visual: Price vs Log Returns

# Ensure datetime index

raw.index = pd.to_datetime(raw.index)

# Create log returns if not already present

if 'log_returns' not in raw.columns:
    raw['log_returns'] = np.log(
        raw['Close'] / raw['Close'].shift(1)
    )

# Remove NaN from shift
returns = raw['log_returns'].dropna()

# Build subplot
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "S&P 500 Close Price",
        "Daily Log Returns"
    )
)

# Top: Price Series
fig.add_trace(
    go.Scatter(
        x=raw.index,
        y=raw['Close'],
        mode='lines',
        name='Close Price'
    ),
    row=1,
    col=1
)

# Bottom: Log Returns
fig.add_trace(
    go.Scatter(
        x=returns.index,
        y=returns,
        mode='lines',
        name='Log Returns'
    ),
    row=2,
    col=1
)

# Zero line for returns
fig.add_hline(
    y=0,
    line_dash='dash',
    opacity=0.6,
    row=2,
    col=1
)

# ADF evidence box
fig.add_annotation(
    x=0.98,
    y=0.32,
    xref='paper',
    yref='paper',
    text=(
        "ADF Statistic: -19.60"
        "<br>P-value: < 0.001"
        "<br>Reject Unit Root"
    ),
    showarrow=False,
    align='left',
    borderwidth=1,
    bgcolor='rgba(30,30,30,0.85)',
    font=dict(size=12)
)

# Layout
fig.update_layout(
    title='From Price Series to Log Returns',
    template='plotly_dark',
    height=850,
    width=1400,
    hovermode='x unified',
    showlegend=False
)

fig.update_yaxes(title_text='Price', row=1, col=1)
fig.update_yaxes(title_text='Log Return', row=2, col=1)
fig.update_xaxes(title_text='Date', row=2, col=1)

# Show
fig.show()

fig.write_image(
     "../reports/figures/stationarity_price_vs_returns.png",
     scale=3
 )

# Rolling Volatility 

In [39]:
raw['rolling_vol_30'] = raw['log_returns'].rolling(window=30).std()

fig = raw['rolling_vol_30'].plot(title='30-Day Rolling Volatility (S&P 500)')
fig.update_layout(
    showlegend=False,
    yaxis_title='Volatility',
    xaxis_title='Date'
)
fig.show()
fig.write_image(
    "../reports/figures/sp500_rolling_volatility.png",
    width=1400,
    height=700,
    scale=2
)

In [40]:

window_size = 30
display(Markdown(f"### 📊 Rolling Volatility ({window_size}-Day)\n\nTo quantify time-varying volatility, a {window_size}-day rolling standard deviation of log returns is computed."))


### 📊 Rolling Volatility (30-Day)

To quantify time-varying volatility, a 30-day rolling standard deviation of log returns is computed.

In [41]:
# === CALENDAR & SEASONALITY ANALYSIS — FRAMING NOTE ===

display(Markdown(f"""
## Calendar and seasonality analysis

The following analysis examines average log returns by day of week and month.
These patterns are included here as descriptive context only. Seasonal
components were formally tested in Notebook 04 using SARIMA — seasonal terms
were found non-significant and AIC was worse than the ARIMA baseline.
Seasonality was rejected as a modelling input on that evidence. The patterns
below are not used in any downstream model.

Dataset period: {raw.index[0].date()} to {raw.index[-1].date()}
Total sessions analysed: {len(raw):,}
"""))


## Calendar and seasonality analysis

The following analysis examines average log returns by day of week and month.
These patterns are included here as descriptive context only. Seasonal
components were formally tested in Notebook 04 using SARIMA — seasonal terms
were found non-significant and AIC was worse than the ARIMA baseline.
Seasonality was rejected as a modelling input on that evidence. The patterns
below are not used in any downstream model.

Dataset period: 2000-01-04 to 2026-06-26
Total sessions analysed: 6,658


In [42]:
# === DRAWDOWN ANALYSIS (High Value for Risk Roles) ===
raw['Return'] = raw['Close'].pct_change()                    # ensure Return column exists
raw['Cumulative'] = (1 + raw['Return']).cumprod()
raw['Peak'] = raw['Cumulative'].cummax()
raw['Drawdown'] = (raw['Cumulative'] - raw['Peak']) / raw['Peak']

print(f"Maximum Drawdown: {raw['Drawdown'].min():.2%}")
print(f"Date of Max Drawdown: {raw['Drawdown'].idxmin().date()}")
print(f"Current Drawdown (as of last date): {raw['Drawdown'].iloc[-1]:.2%}")

# Visualization
fig = raw['Drawdown'].plot(title='S&P 500 Historical Drawdowns')
fig.update_layout(
    yaxis_title='Drawdown (%)',
    xaxis_title='Date',
    showlegend=False
)

fig.show()
fig.write_image(
    "../reports/figures/sp500_drawdowns.png",
    width=1400,
    height=700,
    scale=2
)

# Top 5 worst drawdowns
print("\n=== Top 5 Largest Drawdowns ===")
print(raw['Drawdown'].nsmallest(5))

Maximum Drawdown: -56.78%
Date of Max Drawdown: 2009-03-09
Current Drawdown (as of last date): -3.36%



=== Top 5 Largest Drawdowns ===
Date
2009-03-09   -0.567754
2009-03-05   -0.563908
2009-03-06   -0.563377
2009-03-03   -0.555103
2009-03-02   -0.552235
Name: Drawdown, dtype: float64


In [43]:


max_dd = raw['Drawdown'].min()
max_dd_date = raw['Drawdown'].idxmin().date()
display(Markdown(f"""
### 📉 Drawdown Analysis — Understanding Portfolio Risk

**Key Statistics:**
- **Maximum Drawdown:** {max_dd:.2%} on **{max_dd_date}**
"""))



### 📉 Drawdown Analysis — Understanding Portfolio Risk

**Key Statistics:**
- **Maximum Drawdown:** -56.78% on **2009-03-09**


In [44]:
# === CALENDAR & SEASONALITY ANALYSIS ===
raw['DayOfWeek'] = raw.index.day_name()
raw['Month'] = raw.index.month_name()
raw['Year'] = raw.index.year

# Average return by day of week
day_order = [
    'Monday', 'Tuesday', 'Wednesday',
    'Thursday', 'Friday'
]

dow_returns = (
    raw.groupby('DayOfWeek')['log_returns']
    .mean()
    .reindex(day_order)
)
print("Average Log Return by Day of Week:\n", dow_returns)

# Average return by month
month_order = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

month_returns = (
    raw.groupby('Month')['log_returns']
    .mean()
    .reindex(month_order)
)
print("\nAverage Log Return by Month:\n", month_returns)

# Plot
fig = px.bar(dow_returns.reset_index(), x='DayOfWeek', y='log_returns', 
             title='Average Daily Log Return by Day of Week')
fig.show()


Average Log Return by Day of Week:
 DayOfWeek
Monday       0.000131
Tuesday      0.000514
Wednesday    0.000320
Thursday     0.000241
Friday      -0.000004
Name: log_returns, dtype: float64

Average Log Return by Month:
 Month
January     -0.000037
February    -0.000315
March        0.000328
April        0.000849
May          0.000359
June        -0.000111
July         0.000693
August       0.000044
September   -0.000714
October      0.000534
November     0.000984
December     0.000247
Name: log_returns, dtype: float64


In [45]:


best_dow = dow_returns.idxmax(); best_dow_val = dow_returns.max()
worst_dow = dow_returns.idxmin(); worst_dow_val = dow_returns.min()
best_month = month_returns.idxmax(); best_month_val = month_returns.max()
worst_month = month_returns.idxmin(); worst_month_val = month_returns.min()

display(Markdown(f"""
### 📅 Calendar & Seasonality Analysis

**Average Log Return by Day of Week:**
- **Strongest:** {best_dow} ({best_dow_val:.6f})
- **Weakest:** {worst_dow} ({worst_dow_val:.6f})

**Average Log Return by Month:**
- **Strongest:** {best_month} ({best_month_val:.6f})
- **Weakest:** {worst_month} ({worst_month_val:.6f})
"""))



### 📅 Calendar & Seasonality Analysis

**Average Log Return by Day of Week:**
- **Strongest:** Tuesday (0.000514)
- **Weakest:** Friday (-0.000004)

**Average Log Return by Month:**
- **Strongest:** November (0.000984)
- **Weakest:** September (-0.000714)


In [46]:
# === SAVE CLEANED DATASET FOR FUTURE PHASES ===
os.makedirs('../data', exist_ok=True)


# Core modeling dataset
core_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'log_returns']

# CSV (human-readable)
raw[core_cols].to_csv('../data/sp500_cleaned.csv')

# Parquet (pipeline version)
raw[core_cols].to_parquet('../data/sp500_cleaned.parquet')

# Full enriched dataset
raw.to_parquet('../data/sp500_eda_enriched.parquet')

print("✅ Datasets exported successfully")


✅ Datasets exported successfully


## EDA summary

The exploratory analysis established four properties of the S&P 500 dataset
that directly inform the modelling approach in subsequent notebooks.

**Data integrity:** One zero-volume observation was identified and removed
(2001-09-17, the first trading session following the September 11 attacks,
where reported volume was recorded anomalously as zero). The cleaned dataset
covers 2000-01-01 to present with no missing closing prices or unexplained
gaps in the trading calendar.

**Return transformation:** Closing prices were converted to log returns for
modelling suitability. Log returns are additive across time, approximately
symmetric for small price changes, and better suited to statistical testing
than raw price series.

**Stationarity:** The ADF test confirms log returns are stationary
(ADF = -19.60, p < 0.001). This is a necessary condition for the
time-series models applied in Notebooks 04 and 05.

**Volatility structure:** Log returns exhibit near-zero mean, time-varying
volatility, and persistent volatility clustering visible in rolling measures
and the ACF of squared returns. Fat tails and extreme events are present in
the distribution. These properties motivate the GARCH-based volatility
modelling approach in Notebook 05 over any assumption of constant variance.